# AML MLP Distillation — ONNX-MLIR Friendly

MLP student that mimics a strong LightGBM teacher on IBM's synthetic AML dataset, then exports to ONNX with only standard ops (MatMul, Relu, Sigmoid, etc.) so it can be compiled with ONNX-MLIR.

*Goal:* get closer to the LightGBM PR-AUC while keeping a simple neural net architecture suitable for ONNX-MLIR.

In [4]:
# All the imports (teacher + student + preprocessing)
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import average_precision_score, precision_recall_curve


import lightgbm as lgb

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

RANDOM_STATE = 42

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True

### Data Loading

In [5]:
# Configuration
repo_root = Path("..")  # notebook is in AIModel/
DATA_PATH = repo_root / "Datasets" / "trans_3000p2_list.csv"
TARGET_COL = "Is Laundering"
ALERT_RATE = 0.001  # 0.1% of scored transactions flagged

print("Data path:", DATA_PATH)

Data path: ../Datasets/trans_3000p2_list.csv


In [8]:
#~1:45

# Load and basic cleaning (mirrors baseline LightGBM notebook)
df = pd.read_csv(DATA_PATH)

# Strip whitespace from column names and string values
df.columns = df.columns.str.strip()
for c in df.columns:
    if df[c].dtype == "object":
        df[c] = df[c].astype(str).str.strip()

# Timestamps and numeric amounts
df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
for amt_col in ["Amount Received", "Amount Paid"]:
    df[amt_col] = pd.to_numeric(df[amt_col], errors="coerce")

# Drop rows with missing essentials and clean target
df = df.dropna(subset=["Timestamp", "Amount Received", "Amount Paid", TARGET_COL]).copy()
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce").fillna(0).astype(int)

print(df.head(3))
print("Rows:", len(df), "| positives:", int((df[TARGET_COL] == 1).sum()))

KeyboardInterrupt: 

### Feature Engineering @ Transaction Level

In [ ]:
# Transaction-level engineered features (same as baseline)
def build_tx_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out["fx_spread"] = out["Amount Paid"] - out["Amount Received"]
    out["is_cross_cur"] = (out["Receiving Currency"] != out["Payment Currency"]).astype(int)
    out["same_bank"] = (out["From Bank"] == out["To Bank"]).astype(int)
    out["self_transfer"] = (out["Account"] == out["To Account"]).astype(int)
    out["hour"] = out["Timestamp"].dt.hour
    out["dow"] = out["Timestamp"].dt.dayofweek
    out["is_weekend"] = out["dow"].isin([5, 6]).astype(int)
    return out

df = build_tx_features(df)
df.head(10)

In [ ]:
#~15

# Time-based split: train = first 80%, test = last 20% by Timestamp
split_time = df["Timestamp"].quantile(0.80)
train_df = df[df["Timestamp"] <= split_time].copy()
test_df = df[df["Timestamp"] > split_time].copy()

print("Train rows:", len(train_df), "| Test rows:", len(test_df))
print("Positives (train/test):", int(train_df[TARGET_COL].sum()), int(test_df[TARGET_COL].sum()))

In [ ]:
#~2:25

# Entity aggregates computed on TRAIN only, then merged into both
def entity_aggs(train: pd.DataFrame, key: str, prefix: str) -> pd.DataFrame:
    tmp = train.copy()
    agg_dict = {
        f"{prefix}_tx_count": ("Timestamp", "size"),
        f"{prefix}_amt_paid_mean": ("Amount Paid", "mean"),
        f"{prefix}_amt_recv_mean": ("Amount Received", "mean"),
        f"{prefix}_cross_ccy_rate": ("is_cross_cur", "mean"),
    }
    g = tmp.groupby(key).agg(**agg_dict).reset_index()
    return g

acc_aggs = entity_aggs(train_df, "Account", "acc")
toacc_aggs = entity_aggs(train_df, "To Account", "toacc")
fb_aggs = entity_aggs(train_df, "From Bank", "fb")
tb_aggs = entity_aggs(train_df, "To Bank", "tb")

def merge_aggs(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame
    out = out.merge(acc_aggs, on="Account", how="left")
    out = out.merge(toacc_aggs, on="To Account", how="left")
    out = out.merge(fb_aggs, on="From Bank", how="left")
    out = out.merge(tb_aggs, on="To Bank", how="left")
    return out

train_df = merge_aggs(train_df)
test_df = merge_aggs(test_df)

agg_cols = [c for c in train_df.columns if c.startswith(("acc_", "toacc_", "fb_", "tb_"))]
train_df[agg_cols] = train_df[agg_cols].fillna(0.0)
test_df[agg_cols] = test_df[agg_cols].fillna(0.0)

len(agg_cols), agg_cols[:6]

In [ ]:
# Feature sets (column info) — same as LightGBM baseline
numeric_features = [
    "Amount Received",
    "Amount Paid",
    "fx_spread",
    "is_cross_cur",
    "same_bank",
    "self_transfer",
    "hour",
    "dow",
    "is_weekend",
] + agg_cols

categorical_features = ["Payment Format", "Receiving Currency", "Payment Currency"]

feature_cols = numeric_features + categorical_features

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Sample features:", feature_cols[:8])

#### OPTIONAL (Saving entire engineered set)

In [ ]:
# Save entire test and train (NOT recommended ~10GB)

test_df.to_csv("../datasets/engineered_testing.csv")
train_df.to_csv("../datasets/engineered_training.csv")

#### Quick Load (if saved)

In [6]:
import os
import sys

if (not os.path.isfile("../datasets/engineered_testing.csv") 
    or not os.path.isfile("../datasets/engineered_training.csv")): 
        print("Datasets not found") 
        sys.exit()

test_df = pd.read_csv("../datasets/engineered_testing.csv")
train_df = pd.read_csv("../datasets/engineered_training.csv")
agg_cols = [c for c in train_df.columns if c.startswith(("acc_", "toacc_", "fb_", "tb_"))]


# Feature sets (column info) — same as LightGBM baseline
numeric_features = [
    "Amount Received",
    "Amount Paid",
    "fx_spread",
    "is_cross_cur",
    "same_bank",
    "self_transfer",
    "hour",
    "dow",
    "is_weekend",
] + agg_cols

categorical_features = ["Payment Format", "Receiving Currency", "Payment Currency"]

feature_cols = numeric_features + categorical_features

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Sample features:", feature_cols[:8])



Numeric features: 25
Categorical features: 3
Sample features: ['Amount Received', 'Amount Paid', 'fx_spread', 'is_cross_cur', 'same_bank', 'self_transfer', 'hour', 'dow']


### Training Data Tensors & Preprocessing

In [7]:
#~1:00

# Preprocessor: passthrough numeric, OneHot categorical
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", ohe, categorical_features),
    ],
    remainder="drop",
)

X_train_df = train_df[feature_cols].copy()
X_test_df = test_df[feature_cols].copy()
y_train = train_df[TARGET_COL].astype(int).copy()
y_test = test_df[TARGET_COL].astype(int).copy()

print("Fitting preprocessor on training data...")
X_train_num = preprocess.fit_transform(X_train_df).astype(np.float32)
X_test_num = preprocess.transform(X_test_df).astype(np.float32)

print("Numeric input dimension after preprocessing:", X_train_num.shape[1])
print("Class balance (train): pos=%d / %d (%.4f%%)" % (
    int(y_train.sum()), len(y_train), 100.0 * y_train.mean(),
))

Fitting preprocessor on training data...
Numeric input dimension after preprocessing: 62
Class balance (train): pos=53687 / 36608479 (0.1467%)


### Teacher model (LightGBM) and distillation targets

In [8]:
#~2:40

# LightGBM teacher trained on the same numeric features X_train_num
teacher = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.07,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    class_weight="balanced",
)

print("Training LightGBM teacher...")
teacher.fit(X_train_num, y_train)

teacher_train_probs = teacher.predict_proba(X_train_num)[:, 1].astype(np.float32)
teacher_test_probs = teacher.predict_proba(X_test_num)[:, 1].astype(np.float32)

ap_teacher = average_precision_score(y_test.to_numpy(), teacher_test_probs)
print(f"Teacher Test Average Precision (PR-AUC): {ap_teacher:.4f}")

Training LightGBM teacher...
[LightGBM] [Info] Number of positive: 53687, number of negative: 36554792
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.391761 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4620
[LightGBM] [Info] Number of data points in the train set: 36608479, number of used features: 62
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Teacher Test Average Precision (PR-AUC): 0.2335


### MLP Student Definition

In [9]:
# MLP architecture (logits out; Sigmoid applied externally)
class AMLMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 128, num_hidden_layers: int = 2): # maybe up this??
        super().__init__()
        layers = []
        last_dim = input_dim
        for _ in range(num_hidden_layers):
            layers.append(nn.Linear(last_dim, hidden_dim))
            layers.append(nn.ReLU())
            last_dim = hidden_dim
        layers.append(nn.Linear(last_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class AMLMLPProb(nn.Module):
    """Wrapper that converts logits to probabilities (for ONNX export)."""
    def __init__(self, base: AMLMLP):
        super().__init__()
        self.base = base

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        logits = self.base(x)
        return torch.sigmoid(logits)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AMLMLP(input_dim=X_train_num.shape[1]).to(device)
model

AMLMLP(
  (net): Sequential(
    (0): Linear(in_features=62, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=1, bias=True)
  )
)

### Distillation Training (MLP mimics LightGBM)

In [ ]:
# Student sees teacher probabilities as soft targets; metrics use real labels.
y_train_soft = teacher_train_probs  # float32 in [0, 1]


# Subsample training set for student (~200k rows max)
max_student_rows = 3
if X_train_num.shape[0] > max_student_rows:
    from sklearn.model_selection import train_test_split
    idx, _ = train_test_split(
        np.arange(X_train_num.shape[0]),
        train_size=max_student_rows,
        random_state=RANDOM_STATE,
        stratify=y_train,
    )
    X_train_student = X_train_num[idx]
    y_train_soft_student = y_train_soft[idx]
else:
    X_train_student = X_train_num
    y_train_soft_student = y_train_soft

criterion = nn.BCEWithLogitsLoss()  # no pos_weight; teacher handled imbalance
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

batch_size = 4096
epochs = 3

train_ds = TensorDataset(
    torch.from_numpy(X_train_student.astype(np.float32)),
    torch.from_numpy(y_train_soft_student.astype(np.float32)),
)
val_ds = TensorDataset(
    torch.from_numpy(X_test_num.astype(np.float32)),
    torch.from_numpy(y_test.to_numpy().astype(np.float32)),  # real labels for metrics
)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

print("Epochs:")

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device).view(-1, 1)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * xb.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)

    # Validation: compare to real labels on test set
    model.eval()
    all_probs = []
    all_targets = []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            logits = model(xb)
            probs = torch.sigmoid(logits).view(-1).cpu().numpy()
            all_probs.append(probs)
            all_targets.append(yb.numpy())

    all_probs = np.concatenate(all_probs)
    all_targets = np.concatenate(all_targets)
    ap = average_precision_score(all_targets, all_probs)

    print(f"Epoch {epoch}/{epochs} - train_loss={epoch_loss:.4f} - val_PR_AUC={ap:.4f}")

Testing epochs...


### Evaluation

In [ ]:
# Test set evaluation: PR-AUC and PR curve (student vs real labels)
model.eval()
with torch.no_grad():
    logits_test = model(torch.from_numpy(X_test_num.astype(np.float32)).to(device))
    probs_test = torch.sigmoid(logits_test).view(-1).cpu().numpy()

ap_test = average_precision_score(y_test.to_numpy(), probs_test)
print(f"Student Test Average Precision (PR-AUC): {ap_test:.4f}")

prec, rec, thr = precision_recall_curve(y_test.to_numpy(), probs_test)
plt.figure()
plt.plot(rec, prec, label=f"MLP student (AP={ap_test:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve — MLP Student")
plt.legend()
plt.show()

print("Positives in test:", int(y_test.sum()), "out of", len(y_test))

In [ ]:
# Evaluate at alert budget (top-k by score)
def eval_at_alert_rate(y_true: pd.Series, scores: np.ndarray, rate: float = 0.001):
    n = len(scores)
    k = max(1, int(n * rate))
    order = np.argsort(scores)[::-1]
    top_idx = order[:k]
    tp = int(y_true.iloc[top_idx].sum())
    precision = tp / k
    recall = tp / int(y_true.sum()) if int(y_true.sum()) > 0 else np.nan
    return k, precision, recall

k_s, p_at_k_s, r_at_k_s = eval_at_alert_rate(y_test, probs_test, rate=ALERT_RATE)
print(f"Student Alert rate: {ALERT_RATE*100:.3f}% -> top-{k_s} rows")
print(f"Student Precision@k: {p_at_k_s:.4f} | Recall@k: {r_at_k_s:.4f}")

# (Optional) compare with teacher at the same alert budget
k_t, p_at_k_t, r_at_k_t = eval_at_alert_rate(y_test, teacher_test_probs, rate=ALERT_RATE)
print(f"Teacher Precision@k: {p_at_k_t:.4f} | Recall@k: {r_at_k_t:.4f}")

### Conversion to ONNX (MLP only, standard ops)

In [ ]:
import datetime
import torch.onnx

models_dir = repo_root / "Models"
models_dir.mkdir(parents=True, exist_ok=True)

ts = datetime.datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
onnx_path = models_dir / f"mlp_aml_{ts}.onnx"

# Wrap with Sigmoid so the exported model returns probabilities
wrapped = AMLMLPProb(model.cpu())
wrapped.eval()
dummy_input = torch.zeros(1, X_train_num.shape[1], dtype=torch.float32)

torch.onnx.export(
    wrapped,
    dummy_input,
    str(onnx_path),
    input_names=["input"],
    output_names=["prob"],
    dynamic_axes={"input": {0: "batch"}, "prob": {0: "batch"}},
    opset_version=15,
    use_external_data_format=False,
)

print("Exported MLP ONNX model to:", onnx_path)